In [1]:
import sqlite3
import os
import shutil
import time
import threading
from datetime import datetime, timedelta
import tkinter as tk
from tkinter import messagebox

# Global variables
test_running = False
history_log = []

# Path to Chrome history file
HISTORY_PATH = os.path.expanduser("~/.config/google-chrome/Default/History")

# Function to fetch history for the past 10 minutes
def get_recent_history():
    global history_log
    history_log.clear()
    if os.path.exists(HISTORY_PATH):
        try:
            shutil.copy2(HISTORY_PATH, "History_Copy")
            conn = sqlite3.connect("History_Copy")
            cursor = conn.cursor()
            
            ten_minutes_ago = datetime.utcnow() - timedelta(minutes=10)
            chrome_time = int((ten_minutes_ago - datetime(1601, 1, 1)).total_seconds() * 1e6)
            
            cursor.execute("SELECT url, title FROM urls WHERE last_visit_time > ?", (chrome_time,))
            history_log.extend(cursor.fetchall())
            conn.close()
        except Exception as e:
            history_log.append(("Error fetching history", str(e)))
    else:
        history_log.append(("No history available", "Ensure Chrome is installed and has history."))

def start_history_tracking():
    while test_running:
        get_recent_history()
        time.sleep(60)  # Check history every minute

# Login Function
def login():
    if username_entry.get() and password_entry.get():
        login_window.destroy()
        open_moodle()
    else:
        messagebox.showerror("Login Failed", "Enter both username and password")

# Moodle-like Application
def open_moodle():
    global moodle_window
    moodle_window = tk.Tk()
    moodle_window.title("Moodle Test Portal")
    
    # Set the background color to Royal Blue
    moodle_window.configure(bg="#4169E1")
    
    tk.Label(moodle_window, text="Welcome to the Test Portal", fg="white", bg="#4169E1", font=("Helvetica", 16)).pack(pady=10)
    tk.Button(moodle_window, text="Write Test", command=start_test, bg="#FF69B4", fg="white", font=("Helvetica", 12)).pack(pady=5)
    tk.Button(moodle_window, text="Exit", command=moodle_window.quit, bg="#C8A2D6", fg="white", font=("Helvetica", 12)).pack(pady=5)
    
    moodle_window.mainloop()

# Start the Test
def start_test():
    global test_running, test_window
    test_running = True
    test_window = tk.Toplevel(moodle_window)
    test_window.title("Test in Progress")
    
    # Set the background color to Royal Blue
    test_window.configure(bg="#4169E1")
    
    questions = [
        ("What is 2 + 2?", "4"),
        ("What is the capital of France?", "Paris"),
        ("Who developed Python?", "Guido van Rossum")
    ]
    
    user_answers = []
    for i, (question, _) in enumerate(questions):
        tk.Label(test_window, text=question, fg="white", bg="#4169E1", font=("Helvetica", 12)).pack(pady=5)
        answer_entry = tk.Entry(test_window)
        answer_entry.pack(pady=5)
        user_answers.append((answer_entry, question))
    
    tk.Button(test_window, text="Submit", command=lambda: submit_test(user_answers, questions), bg="#FF69B4", fg="white", font=("Helvetica", 12)).pack(pady=10)
    
    threading.Thread(target=start_history_tracking, daemon=True).start()
    test_window.after(600000, lambda: submit_test(user_answers, questions))  # Auto-submit after 10 minutes

# Submit Test
def submit_test(user_answers, questions):
    global test_running
    if not test_running:
        return  # Ensure that test is not already submitted
    
    test_running = False
    test_window.destroy()  # Close the test window after submission
    
    result_window = tk.Toplevel(moodle_window)
    result_window.title("Test Results")
    
    # Set the background color to Royal Blue
    result_window.configure(bg="#4169E1")
    
    # Safely calculate score
    score = 0
    for entry, (_, correct) in zip(user_answers, questions):
        entry_widget = entry[0]  # Get the actual Entry widget
        if entry_widget.winfo_exists() and entry_widget.get().strip().lower() == correct.lower():
            score += 1
    
    tk.Label(result_window, text=f"Your Score: {score}/{len(questions)}", fg="white", bg="#4169E1", font=("Helvetica", 14)).pack(pady=10)
    
    tk.Label(result_window, text="Browsing History During Test:", fg="white", bg="#4169E1", font=("Helvetica", 12)).pack(pady=5)
    if history_log:
        for url, title in history_log:
            tk.Label(result_window, text=f"{title}: {url}", fg="white", bg="#4169E1", font=("Helvetica", 10)).pack(pady=2)
    else:
        tk.Label(result_window, text="No browsing activity detected.", fg="white", bg="#4169E1", font=("Helvetica", 10)).pack(pady=2)
    
    tk.Button(result_window, text="Close", command=result_window.destroy, bg="#C8A2D6", fg="white", font=("Helvetica", 12)).pack(pady=10)

# Login UI
login_window = tk.Tk()
login_window.title("Login Page")

# Set the background color to Royal Blue
login_window.configure(bg="#4169E1")

tk.Label(login_window, text="Username:", fg="white", bg="#4169E1", font=("Helvetica", 12)).pack(pady=5)
username_entry = tk.Entry(login_window)
username_entry.pack(pady=5)

tk.Label(login_window, text="Password:", fg="white", bg="#4169E1", font=("Helvetica", 12)).pack(pady=5)
password_entry = tk.Entry(login_window, show="*")
password_entry.pack(pady=5)

tk.Button(login_window, text="Login", command=login, bg="#FF69B4", fg="white", font=("Helvetica", 12)).pack(pady=10)

login_window.mainloop()
